# Exercise 1 — Build a Golden Eval Set

You're going to write test cases for a small Q&A bot that answers
questions about a fictional battery startup, **Voltaic Labs**, using a
fixed set of internal documents as its only source of truth.

**The goal isn't to find bugs by accident — it's to write a deliberate,
diverse set of test cases**, the way you would for any app you're
shipping. Aim for 15–20 test cases, with **at least 5 designed to break
the bot** (edge cases, ambiguous questions, questions outside what the
docs cover).

You won't grade these automatically yet — that's Exercise 2. For now,
just run each question through the bot and read the answer yourself.

## Setup

Needs your own Anthropic API key, and the repo cloned so `app.py` and
the shared fixtures are available. Run this once.

In [ ]:
!pip install -q anthropic
!git clone -q https://github.com/PrepVector/Agent_Evals.git repo 2>/dev/null || echo "(repo already cloned)"
%cd repo/workshop_demo_claude/rag_app

import os
# Replace PASTE_KEY_HERE with your API key. Do not share or commit this notebook with your key in it.
os.environ["ANTHROPIC_API_KEY"] = "PASTE_KEY_HERE"
assert os.environ["ANTHROPIC_API_KEY"] != "PASTE_KEY_HERE", "Replace PASTE_KEY_HERE with your API key first."

from app import answer_question, PAGE_CONTENT

print(f"App loaded. {len(PAGE_CONTENT)} documents in the knowledge base:")
for url in PAGE_CONTENT:
    print(" -", url)

## 1. Get familiar with what the bot actually knows

Before writing test cases, skim the source documents — you can only
write a good adversarial question if you know what's *not* in there,
or where two documents might disagree.

In [ ]:
for url, text in PAGE_CONTENT.items():
    print(f"--- {url} ---")
    print(text)
    print()

## 2. Write your test cases

Each test case is a dict with:
- `id` — short unique label
- `question` — what you'll ask the bot
- `category` — a short label for what kind of case this is (see suggestions below)
- `why` — one line on what this case is checking for (you'll want this
  later when comparing notes with a partner)

Three examples are filled in below. **Add at least 12 more** to reach
15–20 total, with at least 5 outside `happy_path`.

Suggested categories to spread your cases across (you don't need all of
them, but aim for more than just happy-path vs. everything-else):
- `happy_path` — a clear question with one unambiguous answer
- `source_conflict` — two documents disagree; does it use the most recent one?
- `out_of_scope` — nothing in the docs covers this; does it say so or guess?
- `compound_question` — asks two things at once; does it answer both?
- `ambiguous` — a vague reference or pronoun with no clear antecedent
- `adversarial` — phrased to nudge the bot toward guessing or complying with something it shouldn't

In [ ]:
TEST_CASES = [
    {
        "id": "happy_1",
        "question": "Who founded Voltaic Labs?",
        "category": "happy_path",
        "why": "Clear, single-fact question with one unambiguous answer in the docs.",
    },
    {
        "id": "conflict_1",
        "question": "What is Voltaic Labs' energy density?",
        "category": "source_conflict",
        "why": "Two documents give different figures (320 vs 410 Wh/kg) -- does the bot pick the most recent one, or just the first one it finds?",
    },
    {
        "id": "scope_1",
        "question": "What is Voltaic Labs' current stock price?",
        "category": "out_of_scope",
        "why": "Not in the docs at all -- does the bot say so, or guess/hallucinate a number?",
    },

    # TODO: add at least 12 more test cases here, following the same format.
    # Aim for 15-20 total, with 5+ outside happy_path.
]

print(f"{len(TEST_CASES)} test cases so far.")


## 3. Run every test case through the bot

This calls the app once per test case and prints the question, category,
and answer so you can read through them.

In [ ]:
for case in TEST_CASES:
    answer = answer_question(case["question"])
    print(f"[{case['category']}] {case['id']}: {case['question']}")
    print(f"  -> {answer}")
    print()

## 4. Compare notes

Pair up with someone near you and compare test sets.

- Did they find a failure mode you didn't think to test?
- For any case where the bot's answer surprised you, could you say in
  one sentence *why* it's wrong — not just that it feels off? (You'll
  need that precision in Exercise 2, when you write a judge that has to
  make the same call automatically.)
- Which of your cases would be easy to grade automatically with a
  simple code check (e.g., "does the answer contain this exact
  number"), and which would need a human — or an LLM judge — to read
  the answer and make a judgment call?